# Phase 3: Feature Engineering

This phase prepares the cleaned dataset for a Linear Regression model. The goal is to convert raw variables such as dates and category-like values into features that the model can learn from.

The workflow is:

Raw Data ? Cleaning ? EDA ? Feature Engineering ? Linear Regression

For the Rossmann-style sales dataset, the main variables include:

- Store
- DayOfWeek
- Date
- Sales
- Customers
- Open
- Promo
- StateHoliday
- SchoolHoliday

We will transform these into a model-ready feature set such as:

- Store
- Customers
- Promo
- SchoolHoliday
- Year
- Month
- Day
- WeekOfYear
- IsWeekend

The target variable remains `Sales`.


In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
train_path = PROJECT_DIR / "data" / "train.csv"

df = pd.read_csv(train_path)
df_ml = df.copy()
df.head()


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,7/31/2015,5263,555,1,1,0,1
1,1,4,7/30/2015,5020,546,1,1,0,1
2,1,3,7/29/2015,4782,523,1,1,0,1
3,1,2,7/28/2015,5011,560,1,1,0,1
4,1,1,7/27/2015,6102,612,1,1,0,1


## 1. Convert the Date column into useful features

The `Date` column is a datetime value, but a raw date is not directly useful to a Linear Regression model. We convert it into time-based variables that capture patterns such as trend and seasonality.


In [ ]:
df_ml["Date"] = pd.to_datetime(df_ml["Date"])

df_ml["Year"] = df_ml["Date"].dt.year
df_ml["Month"] = df_ml["Date"].dt.month
df_ml["Day"] = df_ml["Date"].dt.day
df_ml["WeekOfYear"] = df_ml["Date"].dt.isocalendar().week.astype(int)

df_ml[["Date", "Year", "Month", "Day", "WeekOfYear"]].head()


,Date,Year,Month,Day,WeekOfYear
0,2015-07-31,2015,7,31,31
1,2015-07-30,2015,7,30,31
2,2015-07-29,2015,7,29,31
3,2015-07-28,2015,7,28,31
4,2015-07-27,2015,7,27,31


## 2. Create a weekend indicator

The `DayOfWeek` variable can be converted into a binary feature to separate regular weekdays from weekends.


In [ ]:
df_ml["IsWeekend"] = (df_ml["DayOfWeek"] >= 6).astype(int)

df_ml[["DayOfWeek", "IsWeekend"]].head(10)


,DayOfWeek,IsWeekend
0,5,0
1,4,0
2,3,0
3,2,0
4,1,0
5,7,1
6,6,1
7,5,0
8,4,0
9,3,0


## 3. Investigate `Open` and decide how to model closed stores

If `Open = 0`, the store is closed and sales are usually zero. This creates a trivial pattern that can dominate the model. For a first Linear Regression project, it is often better to focus only on open stores.


In [ ]:
print(df_ml["Open"].value_counts())

df_model = df_ml[df_ml["Open"] == 1].copy()
df_model.head()


Open
1    844392
0    172817
Name: count, dtype: int64


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,Year,Month,Day,WeekOfYear,IsWeekend
0,1,5,2015-07-31,5263,555,1,1,0,1,2015,7,31,31,0
1,1,4,2015-07-30,5020,546,1,1,0,1,2015,7,30,31,0
2,1,3,2015-07-29,4782,523,1,1,0,1,2015,7,29,31,0
3,1,2,2015-07-28,5011,560,1,1,0,1,2015,7,28,31,0
4,1,1,2015-07-27,6102,612,1,1,0,1,2015,7,27,31,0


## 4. Identify categorical variables

Some columns may look numeric but behave as categories. For example, `Store`, `DayOfWeek`, and `StateHoliday` are not naturally ordinal variables. They should be encoded before they can be used in a regression model.


In [ ]:
df_model = pd.get_dummies(
    df_model,
    columns=["DayOfWeek", "StateHoliday"],
    drop_first=True
)

df_model.columns


Index(['Store', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'SchoolHoliday',
       'Year', 'Month', 'Day', 'WeekOfYear', 'IsWeekend', 'DayOfWeek_2',
       'DayOfWeek_3', 'DayOfWeek_4', 'DayOfWeek_5', 'DayOfWeek_6',
       'DayOfWeek_7', 'StateHoliday_a', 'StateHoliday_b', 'StateHoliday_c'],
      dtype='str')

## 5. Prepare the feature matrix and target

We separate the input variables (`X`) from the target variable (`y`).


In [ ]:
X = df_model.drop(columns=["Sales", "Date"])
y = df_model["Sales"]

print("X shape:", X.shape)
print("y shape:", y.shape)
X.head()


X shape: (844392, 19)
y shape: (844392,)


,Store,Customers,Open,Promo,SchoolHoliday,Year,Month,Day,WeekOfYear,IsWeekend,DayOfWeek_2,DayOfWeek_3,DayOfWeek_4,DayOfWeek_5,DayOfWeek_6,DayOfWeek_7,StateHoliday_a,StateHoliday_b,StateHoliday_c
0,1,555,1,1,1,2015,7,31,31,0,False,False,False,True,False,False,False,False,False
1,1,546,1,1,1,2015,7,30,31,0,False,False,True,False,False,False,False,False,False
2,1,523,1,1,1,2015,7,29,31,0,False,True,False,False,False,False,False,False,False
3,1,560,1,1,1,2015,7,28,31,0,True,False,False,False,False,False,False,False,False
4,1,612,1,1,1,2015,7,27,31,0,False,False,False,False,False,False,False,False,False


## 6. Check dtypes and evaluate leakage risk

Before training, confirm the data types and check whether any feature would not be available at prediction time. This is especially important when using variables like `Customers`.


In [30]:
X.dtypes

# IMPORTANT:
# Customers is strongly related to Sales, but it may not be available in a true
# future forecasting scenario. For a first-pass model, this is acceptable for
# learning the pipeline, but it should be revisited when building a real forecast.


Store             int64
Customers         int64
Open              int64
Promo             int64
SchoolHoliday     int64
Year              int32
Month             int32
Day               int32
WeekOfYear        int64
IsWeekend         int64
DayOfWeek_2        bool
DayOfWeek_3        bool
DayOfWeek_4        bool
DayOfWeek_5        bool
DayOfWeek_6        bool
DayOfWeek_7        bool
StateHoliday_a     bool
StateHoliday_b     bool
StateHoliday_c     bool
dtype: object

## 7. Final feature engineering summary

At this stage, the data has been transformed into a model-ready format:

- date-derived features (`Year`, `Month`, `Day`, `WeekOfYear`)
- weekend flag (`IsWeekend`)
- categorical encoding for day and holiday values
- filtered open-store dataset for realistic sales prediction
- inputs `X` and target `y` prepared for regression

The notebook is now ready for the next stage: building and evaluating a Linear Regression model.
